In [2]:
!pip install zarr
!pip install geff

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.9/376.9 kB 7.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 65.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 51.6 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numcodecs
    Found existing installation: numcodecs 0.16.5
    Uninstalling numcodecs-0.16.5:
      Successfully uninstalled numcodecs-0.16.5


In [3]:
import os
import glob
import zarr
import numpy as np
from tqdm.auto import tqdm
import networkx as nx
from geff import read # reading .geff
import glob

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan Device: {device}")
if torch.cuda.is_available():
    print(f"Nama GPU: {torch.cuda.get_device_name(0)}")

Menggunakan Device: cuda
Nama GPU: Tesla T4


In [5]:
COMPETITION_DATA_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development"
TRAIN_DIR = os.path.join(COMPETITION_DATA_DIR, "train")

In [6]:
EPOCHS = 5                  # Berapa kali model melihat seluruh data
BATCH_SIZE = 1              # Gunakan 1 atau 2 agar RAM GPU tidak OOM (Zarr 3D sangat berat)
LEARNING_RATE = 1e-4        # Kecepatan belajar model
NUM_WORKERS = 2             # Jumlah thread CPU untuk memuat data

PATCH_SIZE = (32, 128, 128) # (Z, Y, X)
SIGMA = 2.0                 # Ukuran Gaussian sphere untuk sel (Ground Truth)

# Path untuk menyimpan model terbaik nanti
MODEL_SAVE_PATH = "/kaggle/working/best_unet3d.pth"

In [7]:
def get_start_coords(size, p_size):
    coords = list(range(0, size, p_size))

    # If the last coordinate plus patch size exceeds the boundary,
    # adjust it backwards to exactly fit the edge.
    if coords[-1] + p_size > size:
        coords[-1] = size - p_size

    # no duplicates if the volume size is perfectly divisible
    return sorted(list(set(coords)))

def get_first_array(zarr_node):
    """Mencari array gambar 4D secara otomatis di dalam Zarr Group/Sub-group."""
    if hasattr(zarr_node, "shape"):
        return zarr_node
    if hasattr(zarr_node, "keys"):
        for key in zarr_node.keys():
            item = zarr_node[key]
            array = get_first_array(item)
            if array is not None:
                return array
    return None


class CellPatchDataset(Dataset):

    def __init__(
        self, zarr_paths, geff_paths, patch_size=(32, 128, 128), sigma=2.0
    ):
        self.zarr_paths = zarr_paths
        self.geff_paths = geff_paths
        self.patch_size = patch_size
        self.sigma = sigma
        self.patch_mapping = []

        # 1. OPTIMASI CACHING (Primitive Struct):
        # Menyimpan koordinat sel sebagai dict of numpy array agar DataLoader Multiprocessing
        # tidak memicu kebocoran RAM CPU (OOM) di Kaggle.
        self.node_cache = {}
        for g_path in self.geff_paths:
            graph, _ = read(str(g_path), backend="networkx")
            for node_id, attrs in graph.nodes(data=True):
                key = (str(g_path), attrs["t"])
                if key not in self.node_cache:
                    self.node_cache[key] = []
                self.node_cache[key].append((attrs["z"], attrs["y"], attrs["x"]))

        # Konversi list koordinat menjadi NumPy Array (int32)
        for key in self.node_cache:
            self.node_cache[key] = np.array(
                self.node_cache[key], dtype=np.int32
            )

        pz, py, px = self.patch_size

        for z_path, g_path in zip(self.zarr_paths, self.geff_paths):
            zarr_root = zarr.open(str(z_path), mode="r")
            zarr_array = get_first_array(zarr_root)

            if zarr_array is None:
                raise ValueError(
                    f"Tidak ditemukan array gambar di dalam {z_path}"
                )

            num_t, size_z, size_y, size_x = zarr_array.shape

            z_starts = get_start_coords(size_z, pz)
            y_starts = get_start_coords(size_y, py)
            x_starts = get_start_coords(size_x, px)

            for t in range(num_t):
                for z in z_starts:
                    for y in y_starts:
                        for x in x_starts:
                            self.patch_mapping.append({
                                 "zarr_path": str(z_path),
                                "geff_path": str(g_path),
                                "t": t,
                                "z_start": z,
                                "y_start": y,
                                "x_start": x,
                            })

    def __len__(self):
        return len(self.patch_mapping)

    def _draw_3d_gaussian(self, heatmap, center):
        cz, cy, cx = center
        pz, py, px = heatmap.shape

        sigma_xy = self.sigma # base sigma for x and y

        # Scale down Z sigma based on physical resolution ratio
        sigma_z = self.sigma * (0.40625/1.625) 

        # Calculate bounding box using the maximum sigma
        radius = int(3 * max(sigma_xy, sigma_z)) 

        z_min, z_max = max(0, int(cz - radius)), min(pz, int(cz + radius + 1))
        y_min, y_max = max(0, int(cy - radius)), min(py, int(cy + radius + 1))
        x_min, x_max = max(0, int(cx - radius)), min(px, int(cx + radius + 1))

        zz, yy, xx = np.ogrid[z_min:z_max, y_min:y_max, x_min:x_max]


        # Apply axis-specific variance to the Gaussian distance formula
        dist_sq = ((zz - cz) ** 2) / (sigma_z ** 2) + ((yy - cy) ** 2) / (sigma_xy ** 2) + ((xx - cx) ** 2) / (sigma_xy ** 2)

        gaussian = np.exp(-dist_sq / 2.0)

        heatmap[z_min:z_max, y_min:y_max, x_min:x_max] = np.maximum(
            heatmap[z_min:z_max, y_min:y_max, x_min:x_max], gaussian
        )

    def __getitem__(self, idx):
        info = self.patch_mapping[idx]
        zarr_path = info["zarr_path"]
        geff_path = info["geff_path"]
        t = info["t"]
        z_s, y_s, x_s = info["z_start"], info["y_start"], info["x_start"]
        pz, py, px = self.patch_size

        zarr_root = zarr.open(zarr_path, mode="r")
        zarr_array = get_first_array(zarr_root)

        input_patch = zarr_array[
            t, z_s : z_s + pz, y_s : y_s + py, x_s : x_s + px
        ].astype(np.float32)

        # 2. OPTIMASI NORMALISASI (Percentile Clipping Standar nnU-Net):
        # Memotong outlier 0.5% & 99.5% agar kontras sel konsisten tanpa memperbesar noise background.
        p_low, p_high = np.percentile(input_patch, (0.5, 99.5))
        if p_high > p_low:
            input_patch = np.clip(input_patch, p_low, p_high)
            input_patch = (input_patch - p_low) / (p_high - p_low)
        else:
            input_patch = np.zeros_like(input_patch)

        target_heatmap = np.zeros(self.patch_size, dtype=np.float32)

        # Ambil koordinat sel dari Primitive Cache
        nodes_at_t = self.node_cache.get((geff_path, t), np.array([]))

        for nz, ny, nx in nodes_at_t:
            if (
                (z_s <= nz < z_s + pz)
                and (y_s <= ny < y_s + py)
                and (x_s <= nx < x_s + px)
            ):
                rel_z = nz - z_s
                rel_y = ny - y_s
                rel_x = nx - x_s
                self._draw_3d_gaussian(target_heatmap, (rel_z, rel_y, rel_x))

        # 3. OPTIMASI MEMORY LAYOUT (Contiguous Memory):
        # Menyusun ulang memori array agar transfer CPU-ke-GPU di PyTorch berjalan maksimal.
        input_patch = np.ascontiguousarray(input_patch)
        target_heatmap = np.ascontiguousarray(target_heatmap)

        input_tensor = torch.from_numpy(input_patch).unsqueeze(0).float()
        target_tensor = torch.from_numpy(target_heatmap).unsqueeze(0).float()

        return input_tensor, target_tensor

In [8]:
import torch.nn as nn
import torch.nn.functional as F

class UNet3D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        # ENCODER 
        self.enc1 = self._double_conv(in_channels, 16)
        self.pool1 = nn.MaxPool3d(kernel_size=2, stride=2)

        self.enc2 = self._double_conv(16, 32)
        self.pool2 = nn.MaxPool3d(kernel_size=2, stride=2)

        # BOTTLENECK
        self.bottleneck = self._double_conv(32, 64)

        # DECODER 
        self.upconv2 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.dec2 = self._double_conv(64, 32)

        self.upconv1 = nn.ConvTranspose3d(32, 16, kernel_size=2, stride=2)
        self.dec1 = self._double_conv(32, 16)

        # FINAL OUTPUT
        self.final_conv = nn.Conv3d(16, out_channels, kernel_size=1)

    def _double_conv(self, in_c, out_c):
        # Return an nn.Sequential block
        return nn.Sequential(
            nn.Conv3d(in_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # Pass 'x' through enc1, then pool1
        x1 = self.enc1(x)
        p1 = self.pool1(x1)

        # Pass through enc2, then pool2
        x2 = self.enc2(p1)
        p2 = self.pool2(x2)

        # Pass through the bottleneck
        bn = self.bottleneck(p2)

        # Upsample using upconv2, concatenate with enc2 (skip connection), pass through dec2
        up2 = self.upconv2(bn)
        cat2 = torch.cat([up2, x2], dim=1)
        d2 = self.dec2(cat2)


        # Upsample using upconv1, concatenate with enc1, pass through dec1
        up1 = self.upconv1(d2)
        cat1 = torch.cat([up1, x1], dim=1)
        d1 = self.dec1(cat1)

        # Pass through final_conv and return
        out = self.final_conv(d1)

        return out

In [9]:
def jaccard_loss_per_sample(pred, target, smooth=1e-5):
    # Flatten starting from dimension 1 to keep batches isolated.
    # Flatten mulai dari dimensi 1 untuk menjaga isolasi antar batch
    # Bentuk berubah dari (B, C, Z, Y, X) menjadi (B, N)
    pred_flat = pred.flatten(start_dim=1)
    target_flat = target.flatten(start_dim=1)

    # Calculate intersection PER SAMPLE.
    # Hitung intersection PER SAMPEL (dimensi 1)
    intersection = (pred_flat * target_flat).sum(dim=1)

    # Calculate union PER SAMPLE using .sum(dim=1)
    # Hitung union PER SAMPEL (dimensi 1)
    union = pred_flat.sum(dim=1) + target_flat.sum(dim=1) - intersection

    # Calculate the dice array (shape B) and return its mean()
    # Hitung array dice score (ukuran B), ubah menjadi loss, lalu rata-ratakan
    jaccard_score = (intersection + smooth) / (union + smooth)
    
    return 1.0 - jaccard_score.mean()

In [10]:
train_zarr_paths = sorted(glob.glob(os.path.join(TRAIN_DIR, "*.zarr")))
train_geff_paths = sorted(glob.glob(os.path.join(TRAIN_DIR, "*.geff")))

train_dataset = CellPatchDataset(
    zarr_paths=train_zarr_paths,
    geff_paths=train_geff_paths,
    patch_size=PATCH_SIZE,
    sigma=SIGMA
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS
)

In [11]:
print(f"Total Patch yang akan dilatih: {len(train_dataset)} patches per Epoch.")

Total Patch yang akan dilatih: 159200 patches per Epoch.


In [12]:
model = UNet3D(in_channels=1, out_channels=1).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [13]:
best_loss = float("inf")
CHECKPOINT_PATH = "/kaggle/working/latest_checkpoint.pth"

In [14]:
# inisialisasi GradScaler sebelum loop dimulai
scaler = GradScaler('cuda')

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch_idx, (inputs, targets) in enumerate(progress_bar):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()

        with autocast('cuda'):
            raw_logits = model(inputs)
            predictions = torch.sigmoid(raw_logits)
            loss = jaccard_loss_per_sample(predictions, targets)

        scaler.scale(loss).backward() # scale the loss first then calculate gradient
        scaler.step(optimizer) # update model weights
        scaler.update() # update scale for next batch

        epoch_loss += loss.item()
        progress_bar.set_postfix({"Jaccard Loss": f"{loss.item():.4f}"})

        # last epoch evaluation
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} done | Average Jaccard Loss: {avg_loss: .4f}")

    # Save check point
    checkpoint = {
        'epoch': epoch + 1, 
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(), # save scaler
        'loss': avg_loss,
    }
    
    torch.save(checkpoint, CHECKPOINT_PATH)

    # save the best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        print(f"Performance increasing! Saving the best weights to: {MODEL_SAVE_PATH}")
        torch.save(model.state_dict(), MODEL_SAVE_PATH)

print("Training done! File .pth is ready to used.")
            
            

Epoch 1/5:   0%|          | 0/159200 [00:00<?, ?it/s]

KeyboardInterrupt: 